# Zpracování obrazu: CPU vs CUDA vs OpenCL

Porovnáme zpracování dvou RGB snímků (jednoduchá detekce pohybu - posun jasného obdélníku na tmavém pozadí).

Vstupy mají tři 8bitové RGB kanály. Testujeme malý náhled **64 × 48**, VGA **640 × 480**, Full HD **1920 × 1080**, 4K **3840 × 2160** a 8K **7680 × 4320**.

U RGB vstupů v jednom průchodu převedeme každý pixel na jas `(R + G + B) // 3`, vypočítáme absolutní rozdíl,
použijeme pevný práh 20 pro získání rozdílové masky a zmenšíme masku 4× v každém rozměru. Výstupní pixel je 1,
pokud alespoň jeden pixel jeho 4×4 bloku detekuje pohyb; jinak je 0. Tato binární maska je výstupem programu.

Pokud používáme GPU, nahrávají se celé vstupní snímky, ale stahuje se pouze zmenšená maska (16× méně pixelů).

V Colabu vyber GPU runtime a spouštěj buňky shora. Je nutno nahrát také
soubor `mask_benchmark.py`.

Malý náhled 64 × 48 ukazuje režii GPU při velmi malém objemu práce. Vítěze vždy určuje měření.


Nejprve do dočasného Colab runtime nainstalujeme potřebné balíčky.
Použijeme `pyopencl` pro OpenCL API, `cupy` pro CUDA API.

In [1]:
import importlib.util
import re
import shutil
import subprocess
import sys

assert shutil.which('nvidia-smi'), (
    'Není dostupný NVIDIA GPU runtime (chybí nvidia-smi). '
    'V Colabu otevři Runtime (Běhové prostředí) > Change runtime type (Změnit typ běhového prostředí), '
    'vyber GPU (např. T4), ulož volbu a po připojení spusť notebook znovu od první buňky. '
    'Pokud Colab GPU nepřidělí kvůli limitům nebo nedostupnosti, zkus to později.'
)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'numpy', 'numba', 'pyopencl', 'matplotlib'])
if importlib.util.find_spec('cupy') is None:
    assert shutil.which('nvcc'), (
        'Chybí CUDA Toolkit (příkaz nvcc). V Colabu vyber Runtime > Change runtime type > GPU. '
        'Pokud už GPU používáš, zvol Runtime > Disconnect and delete runtime, znovu se připoj '
        'ke GPU runtime a spusť notebook od první buňky. Smazáním runtime přijdeš o soubory '
        'uložené jen v jeho dočasném úložišti; nejprve si je stáhni.'
    )
    text = subprocess.check_output(['nvcc', '--version'], text=True)
    match = re.search(r'release (\d+)\.', text)
    assert match, (
        'Nelze zjistit verzi CUDA z nvcc --version. Spusť v samostatné buňce !nvcc --version '
        'a zkontroluj výstup. V Colabu zkus nový GPU runtime a spusť notebook od první buňky. '
        f'Výstup nvcc: {text}'
    )
    major = int(match.group(1))
    assert major in (12, 13), (
        f'Zjištěna CUDA {major}; tato buňka podporuje pouze CUDA 12 a 13. '
        'Otevři https://docs.cupy.dev/en/stable/install.html, ověř podporu této verze CUDA '
        'a podle dokumentace uprav výběr balíčku a povolené verze v této buňce. '
        'Pro CUDA 12 použij cupy-cuda12x, pro CUDA 13 cupy-cuda13x; instaluj pouze jednu variantu.'
    )
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'cupy-cuda{major}x'])

In [2]:
from pathlib import Path
import subprocess

cpu_models = sorted({
    line.split(':', 1)[1].strip()
    for line in Path('/proc/cpuinfo').read_text().splitlines()
    if line.startswith('model name')
})
print('CPU:', ', '.join(cpu_models) or 'Model nelze zjistit z /proc/cpuinfo.')
gpu_models = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True
)
for index, model in enumerate(gpu_models.splitlines()):
    print(f'GPU {index}: {model.strip()}')

CPU: Intel(R) Xeon(R) CPU @ 2.00GHz
GPU 0: Tesla T4


In [3]:
import pyopencl as cl
try:
    for p in cl.get_platforms():
        print('Platforma:', p.name)
        for d in p.get_devices():
            print(' ', d.name, '|', cl.device_type.to_string(d.type), '|', d.driver_version)
except cl.Error as error:
    print('OpenCL není dostupné:', error)

Platforma: NVIDIA CUDA
  Tesla T4 | ALL | GPU | 580.82.07


### Volitelná oprava registrace NVIDIA OpenCL v Colabu

Některá prostředí mají OpenCL knihovnu ovladače, ale nemají její ICD registraci. Tato buňka je
ve výchozím stavu vypnutá. Při chybě `PLATFORM_NOT_FOUND_KHR` můžeš nastavit `OPRAVIT_ICD = True`.
Použije pouze knihovnu skutečně nalezenou v runtime a vytvoří místní ICD soubor; neinstaluje ovladače.
Pokud knihovna chybí, přeskoč OpenCL nebo notebook spusť lokálně s podporovaným ovladačem.


In [4]:
OPRAVIT_ICD = False
if OPRAVIT_ICD:
    import ctypes.util
    import os
    from pathlib import Path
    library = ctypes.util.find_library('nvidia-opencl')
    if library is None:
        print('NVIDIA OpenCL knihovna nebyla nalezena. OpenCL backend bude vynechán.')
    else:
        folder = Path('opencl_icd').resolve()
        folder.mkdir(exist_ok=True)
        (folder / 'nvidia.icd').write_text(library + '\n', encoding='utf-8')
        os.environ['OCL_ICD_VENDORS'] = str(folder)
        print('ICD pro nový proces benchmarku:', library)


## Benchmark

Pro každé rozlišení provedeme 3 zahřátí a 20 měření.
CPU používá jednovláknovou kompilovanou smyčku; GPU neupřednostňujeme umělým zpomalením CPU.

**Celek rezidentní** měří výpočet nad vstupními daty již v paměti zařízení.

**Celek CPU→CPU** zahrnuje nahrání obou vstupních snímků i stažení zmenšené binární masky.

Časy zahrnují Python API, alokace a čekání na GPU; nejde o izolované časy kernelů.
RGB používá tři uint8 kanály s celočíselným průměrem. Zrychlení pod 1× znamená, že CPU je rychlejší včetně přenosů.


In [ ]:
OPENCL_DEVICE = None  # při více GPU zadej například '140T' nebo název NVIDIA zařízení
command = [sys.executable, '-u', 'mask_benchmark.py', '--backends', 'cpu', 'cuda', 'opencl',
           '--repeats', '20', '--warmup', '3', '--output', 'mask_benchmark']
if OPENCL_DEVICE:
    command += ['--opencl-device', OPENCL_DEVICE]
subprocess.run(command, check=True)

In [ ]:
from html import escape
from itertools import groupby
from IPython.display import display, HTML, Markdown

report = Path('mask_benchmark/vysledky.md').read_text(encoding='utf-8')
lines = report.splitlines()
table_lines = [line for line in lines if line.startswith('|')]
headers = [value.strip() for value in table_lines[0].strip('|').split('|')]
rows = [[value.strip() for value in line.strip('|').split('|')] for line in table_lines[2:]]

display(Markdown('\n'.join(lines[:lines.index(table_lines[0])])))
display(Markdown('Zeleně: nejrychlejší backend podle mediánu **Celek CPU→CPU** (včetně přenosů). Při shodě jsou zvýrazněny všechny nejrychlejší řádky.'))
html = '<table style="border-collapse: collapse;"><thead><tr>'
html += ''.join(f'<th style="padding: 8px;">{escape(value)}</th>' for value in headers)
html += '</tr></thead>'
for _, group in groupby(rows, key=lambda row: row[0]):
    group = list(group)
    fastest = min(float(row[3]) for row in group)
    html += '<tbody>'
    for index, row in enumerate(group):
        html += '<tr>'
        for column, value in enumerate(row):
            style = 'padding: 8px;'
            if index == 0:
                style += 'border-top: 3px solid #666;'
            if index == len(group) - 1:
                style += 'border-bottom: 3px solid #666;'
            if column == 0:
                style += 'border-left: 3px solid #666;'
            if column == len(row) - 1:
                style += 'border-right: 3px solid #666;'
            if float(row[3]) == fastest:
                style += 'background-color: #c6efce; color: #006100;'
            html += f'<td style="{style}">{escape(value)}</td>'
        html += '</tr>'
    html += '</tbody>'
html += '</table>'
display(HTML(html))
